# CloudBQ pass 2 — L2 likelihood tiers: the lab (Vertex AI Workbench)

The first layer (L1, `scoring/hot_score_v1`) scores a new deal on the completeness of its information. This second layer (L2) classifies
**every open deal of every pipeline** by its likelihood to close, from the certainty each pipeline actually records, into bins that become
tiers. This notebook is lane **N**, the lab: it takes a snapshot of the CRM through the function's own HubSpot client, measures fill rates
and distributions, bins with `numpy.digitize`, calibrates the bins against what really closed, and proves per deal that BigQuery's
`RANGE_BUCKET` gives the same bin. Lane **Q** (production) is `sql/deal_tiers_v1.sql` + `sql/deal_tiers_apply.sql` over the daily
HubSpot → BigQuery sync; lane **W** (event) is the `deal.propertyChange` webhook. The constants live once in
`scoring/deal_tiers_v1.constants.json`. This notebook writes `deal_tiers_lab` and never the gold table `deal_tiers`.

What the CRM said on 2026-10-16 (hub 9201667), which fixed four decisions below:

| Fact | Consequence |
|---|---|
| `icalps_oppocertainty` (number) is filled on 799 of 800 Icalps deals; `icalps_dealcertainty` is a *string* with no values | Icalps certainty = `icalps_oppocertainty`; the string property is unusable |
| every other pipeline has only `hs_deal_stage_probability` (stage-derived: 0.1 / 0.2 / 0.5 / 1.0 on SealSQ); `win_probability` mirrors it; `hs_forecast_probability` is empty | SealSQ certainty = the stage probability; manual edits (0.3, 0.7, 0.75, 0.9 on a handful of deals) are the only values between 0.5 and 1.0 |
| both properties are **fractions 0–1** | the operator's 50 % / 75 % / 90 % are the edges 0.5 / 0.75 / 0.9 |
| 325 of 325 Icalps closed-won deals carry certainty 1.0; losers keep their last value | the snapshot cannot calibrate winners; calibration uses the stage-entry dates `hs_v2_date_entered_<stage>` instead |
| six open Icalps deals sit at exactly 0.49 | a 0.5 edge is already a behavioural boundary for Icalps users |

Decisions taken as defaults (change them in the constants file): below 0.5 → `Tier4_pipeline`; closed deals → `Closed_won` / `Closed_lost`,
never a tier; the same edges for every pipeline, knowing that SealSQ's stage probabilities make `Tier2_opportunity` reachable only through a
manual probability edit.

## 0. Settings, constants, the binning function

In [ ]:
import os, re, sys, json, shlex, pathlib, getpass
from datetime import datetime, timedelta, timezone
import numpy as np, pandas as pd
from IPython.display import display

_cwd = pathlib.Path.cwd()
REPO = _cwd.parent if (_cwd.name == "notebooks" and (_cwd.parent / ".git").exists()) else pathlib.Path.home() / "CloudBQ"
assert (REPO / ".git").exists(), f"clone the repo first (see notebooks/README.md): {REPO}"
os.chdir(REPO)

def load_env(path):
    env = {}
    for line in path.read_text().splitlines():
        parts = shlex.split(line, comments=True)
        if parts and "=" in parts[0]:
            k, v = parts[0].split("=", 1); env[k] = v
    return env
ENV = load_env(REPO / ".env") if (REPO / ".env").exists() else {}
PROJECT = ENV.get("PROJECT") or os.environ.get("PROJECT") or input("GCP project: ")
DATASET = ENV.get("DATASET") or "hubspot"; BQ_LOCATION = ENV.get("BQ_LOCATION") or "EU"
assert not PROJECT.startswith(("your-", "<")), "PROJECT still holds the placeholder"

K = json.loads((REPO / "scoring" / "deal_tiers_v1.constants.json").read_text())
EDGES = np.array(K["edges"], dtype=float); LABELS = K["labels"]; CLOSED = K["closed_labels"]; SOURCE = K["certainty_source"]
print("project", PROJECT, "| dataset", DATASET, "| edges", EDGES.tolist(), "| labels", LABELS)

def bin_index(certainty):
    """numpy.digitize with the constants' edges: 0 below EDGES[0], k when EDGES[k-1] <= x < EDGES[k], len(EDGES) at or above the last edge.
    Identical to BigQuery RANGE_BUCKET(certainty, EDGES) — section 4 proves it per deal. NaN stays NaN (no tier)."""
    x = pd.to_numeric(certainty, errors="coerce")
    idx = pd.Series(np.digitize(x.fillna(-1.0), EDGES), index=x.index).astype("float")
    return idx.mask(x.isna())

def tier(certainty, is_closed, is_closed_won):
    idx = bin_index(certainty)
    out = idx.map(lambda i: None if pd.isna(i) else LABELS[int(i)])
    return out.mask(is_closed.fillna(False).astype(bool), CLOSED["lost"]).mask(is_closed_won.fillna(False).astype(bool), CLOSED["won"])

demo = pd.DataFrame({"certainty": [0.0, 0.49, 0.5, 0.74, 0.75, 0.89, 0.9, 1.0, None]})
demo["bin_index"] = bin_index(demo["certainty"]); demo["tier"] = tier(demo["certainty"], pd.Series(False, index=demo.index), pd.Series(False, index=demo.index))
display(demo)

## 1. The snapshot — every deal through the function's HubSpot client

The token comes from Secret Manager (`hubspot-token`, created by lane A's `secret` step) or, failing that, a masked prompt; it is
never written anywhere. Properties: the function's list plus the certainty candidates, the closed flags, and one stage-entry date
per stage of every pipeline (`hs_v2_date_entered_<stage id>`), which the calibration needs.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "cloud_function_new_deals" / "requirements.txt"), "db-dtypes", "pyarrow", "pandas", "numpy", "requests"], check=False)   # main.py imports functions_framework; to_dataframe needs db-dtypes
sys.path.insert(0, str(REPO / "cloud_function_new_deals")); import main, requests
try:
    from google.cloud import secretmanager
    token = secretmanager.SecretManagerServiceClient().access_secret_version(name=f"projects/{PROJECT}/secrets/hubspot-token/versions/latest").payload.data.decode()
    print("token: Secret Manager hubspot-token")
except Exception as e:
    print(f"Secret Manager unavailable ({type(e).__name__}); prompting"); token = getpass.getpass("HubSpot private-app token (read scopes): ")
hs = main.HubSpot(token); del token

pipelines, stages = hs.pipelines()                                   # id → label
_pl = requests.get(f"{main.HS}/crm/v3/pipelines/deals", headers=hs.h, timeout=30); _pl.raise_for_status()
stage_prob = {st["id"]: float(st["metadata"]["probability"]) for p_ in _pl.json()["results"] for st in p_["stages"]}   # the stage's CONFIGURED probability
stage_pipeline = {st["id"]: p_["id"] for p_ in _pl.json()["results"] for st in p_["stages"]}
stage_entry_props = [f"hs_v2_date_entered_{sid}" for sid in stages]  # one per stage of every pipeline
EXTRA = ["icalps_oppocertainty", "icalps_dealcertainty", "win_probability", "hs_forecast_probability", "hs_is_closed", "hs_is_closed_won", "hs_lastmodifieddate"]
main.PROPS = list(dict.fromkeys(main.PROPS + EXTRA + stage_entry_props))   # search_created() reads the module-level list
raw = hs.search_created(datetime(2000, 1, 1, tzinfo=timezone.utc), datetime.now(timezone.utc))   # HubSpot search caps at 10,000 results; the portal has ~2,000 deals
deals = pd.DataFrame([{"deal_id": str(d["id"]), **d.get("properties", {})} for d in raw])
deals["pipeline_label"] = deals["pipeline"].map(pipelines).fillna(deals["pipeline"]); deals["stage_label"] = deals["dealstage"].map(stages).fillna(deals["dealstage"])
for col in ("hs_deal_stage_probability", "icalps_oppocertainty", "win_probability", "hs_forecast_probability", "amount_in_home_currency", "icalps_dealcertainty"):
    deals[col] = pd.to_numeric(deals[col], errors="coerce") if col in deals and col != "icalps_dealcertainty" else (deals[col] if col in deals else np.nan)   # a property with no value anywhere may be absent from the response
for col in ("hs_is_closed", "hs_is_closed_won"):
    deals[col] = deals[col].map({"true": True, "false": False}) if col in deals else False
for col in stage_entry_props:
    if col in deals: deals[col] = deals[col].replace("", np.nan)         # an unset date must not count as "entered"
print(len(deals), "deals;", deals["pipeline_label"].value_counts().to_dict())

## 2. Fill rates and distributions of the certainty candidates, per pipeline

In [ ]:
cands = ["hs_deal_stage_probability", "icalps_oppocertainty", "win_probability", "hs_forecast_probability", "icalps_dealcertainty"]
fill = deals.groupby("pipeline_label")[cands].apply(lambda g: g.notna().mean().round(3)); fill["deals"] = deals.groupby("pipeline_label").size()
display(fill)
open_deals = deals[deals["hs_is_closed"] == False]
print("open deals:", len(open_deals))
display(open_deals.groupby(["pipeline_label", "hs_deal_stage_probability"]).size().unstack(fill_value=0))
display(open_deals[open_deals["pipeline"] == "766126206"]["icalps_oppocertainty"].value_counts().sort_index().to_frame("open Icalps deals"))

## 3. Certainty per pipeline → bins → tiers (the same CASE as `deal_certainty_v1` / `deal_tiers_v1`)

In [ ]:
deals["certainty_source"] = np.where(deals["pipeline"].isin([k for k in SOURCE if k != "default"]), deals["pipeline"].map(SOURCE), SOURCE["default"])
deals["certainty"] = np.where(deals["certainty_source"] == "icalps_oppocertainty", deals["icalps_oppocertainty"], deals["hs_deal_stage_probability"])
deals["bin_index"] = bin_index(deals["certainty"])
deals["tier"] = tier(deals["certainty"], deals["hs_is_closed"], deals["hs_is_closed_won"])
summary = deals.groupby(["pipeline_label", "tier"], dropna=False).agg(deals=("deal_id", "size"), amount_kusd=("amount_in_home_currency", "sum")).reset_index()
display(summary.pivot(index="pipeline_label", columns="tier", values="deals").fillna(0).astype(int))
display(deals[deals["hs_is_closed"] == False].groupby(["pipeline_label", "tier"], dropna=False).size().unstack(fill_value=0))

## 4. Calibration — does a higher certainty close more often?

Winners' certainty is overwritten at close (Icalps) or equals 1.0 by stage (SealSQ), so the snapshot cannot calibrate on the certainty
value itself. The stage-entry dates can: for every pipeline and stage, how many deals ever entered it, and how many of those are won,
lost or still open. The won share among closed deals is the empirical conversion the stage probability claims.

In [ ]:
rows = []
for pid, plabel in pipelines.items():
    sub = deals[deals["pipeline"] == pid]
    for sid, slabel in stages.items():
        if stage_pipeline.get(sid) != pid: continue                      # a stage belongs to one pipeline
        col = f"hs_v2_date_entered_{sid}"
        if col not in sub or sub[col].notna().sum() == 0: continue
        entered = sub[sub[col].notna()]
        won = int((entered["hs_is_closed_won"] == True).sum()); closed = int((entered["hs_is_closed"] == True).sum()); lost = closed - won
        rows.append({"pipeline": plabel, "stage": slabel, "stage_probability": stage_prob.get(sid),   # the stage's configured probability, not the entrants' current one
                     "entered": len(entered), "open": len(entered) - closed, "won": won, "lost": lost, "won_of_closed": round(won / closed, 3) if closed else None})
calib = pd.DataFrame(rows).sort_values(["pipeline", "entered"], ascending=[True, False])
display(calib)

## 5. Parity with BigQuery — `RANGE_BUCKET` per deal, and the lab table

The snapshot goes to `deal_tiers_lab` (truncated on every run; the production gold table `deal_tiers` is written only by
`sql/deal_tiers_apply.sql`). BigQuery then bins the same rows with `RANGE_BUCKET(certainty, edges)` and every `bin_index` must equal
numpy's. The job id, bytes and slot time are printed so the execution is witnessed, not assumed.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location=BQ_LOCATION)
LAB = f"{PROJECT}.{DATASET}.deal_tiers_lab"
lab = deals[["deal_id", "pipeline", "dealstage", "dealname", "certainty_source", "certainty", "bin_index", "tier", "hs_is_closed", "hs_is_closed_won", "amount_in_home_currency"]].rename(
    columns={"pipeline": "pipeline_id", "dealstage": "dealstage_id", "hs_is_closed": "is_closed", "hs_is_closed_won": "is_closed_won", "amount_in_home_currency": "amount_kusd"}).copy()
lab["computed_at"] = pd.Timestamp.now(tz="UTC"); lab["orchestrator"] = "workbench-notebook-snapshot"; lab["bin_index"] = lab["bin_index"].astype("Int64")
job = client.load_table_from_dataframe(lab, LAB, job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")); job.result()
print(f"loaded {job.output_rows} rows into {LAB} (job {job.job_id})")

edges_sql = "[" + ", ".join(repr(float(e)) for e in EDGES) + "]"
sql = f"SELECT deal_id, certainty, bin_index AS numpy_bin, RANGE_BUCKET(certainty, {edges_sql}) AS bq_bin FROM `{LAB}`"
print(sql); qj = client.query(sql); cmp = qj.result().to_dataframe()
print(f"job {qj.job_id} | bytes processed {qj.total_bytes_processed} | slot ms {qj.slot_millis} | cache hit {qj.cache_hit}")
same = ((cmp["numpy_bin"].isna() & cmp["bq_bin"].isna()) | (cmp["numpy_bin"] == cmp["bq_bin"])).fillna(False).astype(bool)   # a NULL on one side only must fail, not be skipped
assert same.all(), cmp[~same]
print(f"RANGE_BUCKET == numpy.digitize on all {len(cmp)} deals")

## 6. Equality with the production gold table, when it exists (lane Q after the first sync and MERGE)

In [ ]:
from google.api_core.exceptions import NotFound
try:
    gold = client.query(f"SELECT pipeline_id, tier, COUNT(*) AS gold_deals FROM `{PROJECT}.{DATASET}.deal_tiers` WHERE tier IS NOT NULL GROUP BY pipeline_id, tier").result().to_dataframe()
except NotFound:
    gold = None; print("deal_tiers does not exist yet: run sql/apply.sh (step 4) after the first HubSpot → BigQuery transfer (step 3)")
if gold is not None and gold.empty:
    print("deal_tiers exists but holds no rows: run sql/deal_tiers_apply.sql (step 8) and re-run this cell")
elif gold is not None:
    labn = client.query(f"SELECT pipeline_id, tier, COUNT(*) AS lab_deals FROM `{LAB}` WHERE tier IS NOT NULL GROUP BY pipeline_id, tier").result().to_dataframe()
    eq = labn.merge(gold, on=["pipeline_id", "tier"], how="outer")
    eq[["lab_deals", "gold_deals"]] = eq[["lab_deals", "gold_deals"]].fillna(0).astype(int); eq["diff"] = eq["lab_deals"] - eq["gold_deals"]
    display(eq.sort_values(["pipeline_id", "tier"]))
    print("differences come from deals created, edited, deleted or un-tiered between the sync (02:00 UTC) and this snapshot — the retired rows (tier NULL) are excluded on both sides")

## 7. Hand-off to the CRM — the property StackSync will fill

StackSync never creates a property definition (mir-load `hs-props` pattern): create `deal_tier_v1` (enumeration, options below, in the
built-in group `dealinformation` like pass 1's `writeback/bootstrap_property.sh`) once with a schema-write token, then map
`deal_tiers.deal_id` → Deals · Record ID and `deal_tiers.tier` → `deal_tier_v1`, direction BigQuery → HubSpot. A retired row (tier NULL)
clears the property on a deal that still exists.

In [ ]:
prop = K["hubspot_property"]
print(json.dumps({"name": prop["name"], "label": prop["label"], "groupName": prop["group"], "type": prop["type"], "fieldType": prop["field_type"], "description": prop["description"],
                  "hasUniqueValue": False, "formField": False, "options": [{"label": o, "value": o, "displayOrder": i} for i, o in enumerate(prop["options"])]}, indent=1))
mapping = pd.DataFrame([{"object_type": "deals", "bigquery_table": f"{PROJECT}.{DATASET}.deal_tiers", "bigquery_column": "deal_id", "hubspot_property": "hs_object_id", "kind": "match_key"},
                        {"object_type": "deals", "bigquery_table": f"{PROJECT}.{DATASET}.deal_tiers", "bigquery_column": "tier", "hubspot_property": prop["name"], "kind": "property"}])
display(mapping)